In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

In [ ]:
ibis_seg = pd.read_csv('ibis_snp_panel_segment_results.csv')

ibis_coef = pd.read_csv('ibis_snp_panel_kinship_results.csv')

ibis_seg

In [ ]:
ibis_coef

In [ ]:
ibis_coef['RC'].value_counts()

# First Order 

In [ ]:
fo_rels = ['Parent Child', 'Full sibling', 'Half Sibling']

ibis_kinship_fo = ibis_coef[ibis_coef['RC'].isin(fo_rels)]
ibis_kinship_fo

fig = px.violin(ibis_kinship_fo, y="Kinship_Coefficient", x="RC", color='pop',
                hover_data=ibis_kinship_fo.columns, template='simple_white')
fig.update_layout(xaxis_title=' ')
fig.show()

# Cousins

In [ ]:
cuz_rels = ['Full First Cousin', 'Full Second Cousin', 'Full Third Cousin', 'Half Forth Cousin', 'Full Fifth Cousin']


ibis_kinship_cuz = ibis_coef[ibis_coef['RC'].isin(cuz_rels)]

fig = px.violin(ibis_kinship_cuz, y="Kinship_Coefficient", x="RC", color='pop',
                hover_data=ibis_kinship_cuz.columns, template='simple_white')
fig.update_layout(xaxis_title=' ')
fig.show()

# Grandparents

In [ ]:
grand_rels = ['Grandparent', 'Great Grandparent', 'Great Great Grandparent', 'Great Great Great Grandparent']


ibis_kinship_grand = ibis_coef[ibis_coef['RC'].isin(grand_rels)]

fig = px.violin(ibis_kinship_grand, y="Kinship_Coefficient", x="RC", color='pop',
                hover_data=ibis_kinship_grand.columns, template='simple_white')
fig.update_layout(xaxis_title=' ')
fig.show()

# Half Cousins

In [ ]:
half_cuz_rels = ['Half First Cousin', 'Half Second Cousin', 'Half Third Cousin', 'Half Forth Cousin', 'Half Fifth Cousin']

ibis_kinship_half_cuz = ibis_coef[ibis_coef['RC'].isin(half_cuz_rels)]

fig = px.violin(ibis_kinship_half_cuz, y="Kinship_Coefficient", x="RC", color='pop',
                hover_data=ibis_kinship_half_cuz.columns, template='simple_white')
fig.update_layout(xaxis_title=' ')
fig.show()

# Now we need to collapse individuals segment estimates

In [ ]:
ibis_seg = pd.read_csv('ibis_snp_panel_segment_results.csv')

In [ ]:
ibis_seg

In [ ]:
keys = ["pop", "seed", "Fam_ID", "Individual1", "Individual2"]

# Make the pair order-invariant, in case (1, 4) ever shows up as (4, 1)
a, b = ibis_seg["Individual1"].to_numpy(), ibis_seg["Individual2"].to_numpy()
ibis_seg["Individual1"], ibis_seg["Individual2"] = np.minimum(a, b), np.maximum(a, b)

# Split length by IBD type up front (much faster than a pivot on 18M rows)
is2 = ibis_seg["IBD_Type"].eq("IBD2")
ibis_seg["ibd1_cM"] = ibis_seg["Genetic_Seg_Length"].where(~is2, 0.0)
ibis_seg["ibd2_cM"] = ibis_seg["Genetic_Seg_Length"].where(is2, 0.0)
ibis_seg["is_ibd2"] = is2

ibis_seg_new = (
    ibis_seg.groupby(keys, sort=False, observed=True)
      .agg(
          RT=("RT", "first"),
          RC=("RC", "first"),
          MC=("MC", "first"),
          GD=("GD", "first"),
          n_seg=("Genetic_Seg_Length", "size"),
          n_ibd2_seg=("is_ibd2", "sum"),
          total_cM=("Genetic_Seg_Length", "sum"),
          ibd1_cM=("ibd1_cM", "sum"),
          ibd2_cM=("ibd2_cM", "sum"),
          max_seg_cM=("Genetic_Seg_Length", "max"),
          n_markers=("Marker_Count", "sum"),
      )
      .reset_index()
)

ibis_seg_new['RC'].value_counts()

## First Order Relationships

In [ ]:
fo_rels = ['Parent Child', 'Full sibling', 'Half Sibling']

ibis_seg_fo = ibis_seg_new[ibis_seg_new['RC'].isin(fo_rels)]


fig = px.violin(ibis_seg_fo, y="total_cM", x="RC", color='pop',
                hover_data=ibis_seg_fo.columns, template='simple_white')
fig.update_layout(xaxis_title=' ')
fig.show()

## Cousin Relationships

In [ ]:
cuz_rels = ['Full First Cousin', 'Full Second Cousin', 'Full Third Cousin', 'Half Forth Cousin', 'Full Fifth Cousin']

ibis_seg_cuz = ibis_seg_new[ibis_seg_new['RC'].isin(cuz_rels)]


fig = px.violin(ibis_seg_cuz, y="total_cM", x="RC", color='pop',
                hover_data=ibis_seg_cuz.columns, template='simple_white')
fig.update_layout(xaxis_title=' ')
fig.show()

## Grandparent Relationships

In [ ]:
grand_rels = ['Grandparent', 'Great Grandparent', 'Great Great Grandparent', 'Great Great Great Grandparent']

ibis_seg_grand = ibis_seg_new[ibis_seg_new['RC'].isin(grand_rels)]


fig = px.violin(ibis_seg_grand, y="total_cM", x="RC", color='pop',
                hover_data=ibis_seg_grand.columns, template='simple_white')
fig.update_layout(xaxis_title=' ')
fig.show()

## Half Cousin Relationships

In [ ]:
half_cuz_rels = ['Half First Cousin', 'Half Second Cousin', 'Half Third Cousin', 'Half Forth Cousin', 'Half Fifth Cousin']

ibis_seg_half_cuz = ibis_seg_new[ibis_seg_new['RC'].isin(half_cuz_rels)]


fig = px.violin(ibis_seg_half_cuz, y="total_cM", x="RC", color='pop',
                hover_data=ibis_seg_half_cuz.columns, template='simple_white')
fig.update_layout(xaxis_title=' ')
fig.show()